# 06 — Model Comparison and Selection

In [1]:
import os
from pathlib import Path
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)

# --- Project Path Setup ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

if not os.path.exists("data/processed/machine_failure_features.csv"):
    raise FileNotFoundError("Features dataset 'data/processed/machine_failure_features.csv' not found! Please run '03_feature_engineering.ipynb' first.")
if not os.path.exists("models/random_forest.pkl"):
    raise FileNotFoundError("Random Forest model 'models/random_forest.pkl' not found! Please run '04_random_forest.ipynb' first.")
if not os.path.exists("models/xgboost.pkl"):
    raise FileNotFoundError("XGBoost model 'models/xgboost.pkl' not found! Please run '05_xgboost.ipynb' first.")

# 1. Load data and recreate the exact same train/test split
data = pd.read_csv("data/processed/machine_failure_features.csv")
X = data.drop(columns=["Machine failure"])
y = data["Machine failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Test set failures to detect:", y_test.sum())

# 2. Load the trained models from disk (no retraining)
rf_model = joblib.load("models/random_forest.pkl")
xgb_model = joblib.load("models/xgboost.pkl")

# 3. Evaluate both models on the test set
results = {}
for name, model in [("Random Forest", rf_model), ("XGBoost", xgb_model)]:
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    results[name] = {
        "Accuracy": round(accuracy_score(y_test, pred), 4),
        "Precision": round(precision_score(y_test, pred), 4),
        "Recall": round(recall_score(y_test, pred), 4),
        "F1 Score": round(f1_score(y_test, pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, prob), 4),
        "False Alarms": int(fp),
        "Missed Failures": int(fn),
    }

# 4. Display comparison table
comparison_df = pd.DataFrame(results)
print("\n--- Model Comparison Table (Test Set, 68 Real Failures) ---")
print(comparison_df)

# 5. Automated Model Selection Rule:
# Higher Recall is preferred; tiebreaker is F1 Score.
rf_recall = results["Random Forest"]["Recall"]
rf_f1 = results["Random Forest"]["F1 Score"]
xgb_recall = results["XGBoost"]["Recall"]
xgb_f1 = results["XGBoost"]["F1 Score"]

if (rf_recall, rf_f1) >= (xgb_recall, xgb_f1):
    selected_model = "random_forest"
else:
    selected_model = "xgboost"

print(f"\nModel Selection Summary:")
print(f"Random Forest -> Recall: {rf_recall}, F1 Score: {rf_f1}")
print(f"XGBoost       -> Recall: {xgb_recall}, F1 Score: {xgb_f1}")
print(f"Selected Winner: {selected_model}")

# 6. Save selection to models/selected_model.txt
os.makedirs("models", exist_ok=True)
with open("models/selected_model.txt", "w") as f:
    f.write(selected_model)

# Verify the file was created
selection_saved = os.path.exists("models/selected_model.txt")
print("\nFile exists check:", selection_saved)
if not selection_saved:
    raise FileNotFoundError("Failed to save 'models/selected_model.txt'.")
print("Saved selection to: models/selected_model.txt")

Test set failures to detect: 68



--- Model Comparison Table (Test Set, 68 Real Failures) ---
                 Random Forest  XGBoost
Accuracy                0.9920   0.9880
Precision               0.9483   0.8333
Recall                  0.8088   0.8088
F1 Score                0.8730   0.8209
ROC-AUC                 0.9711   0.9856
False Alarms            3.0000  11.0000
Missed Failures        13.0000  13.0000

Model Selection Summary:
Random Forest -> Recall: 0.8088, F1 Score: 0.873
XGBoost       -> Recall: 0.8088, F1 Score: 0.8209
Selected Winner: random_forest

File exists check: True
Saved selection to: models/selected_model.txt
